# Задача: Анализ текста о Data Science

В этом примере давайте выполним простое упражнение, охватывающее все этапы традиционного процесса Data Science. Вам не нужно писать код, вы можете просто нажимать на ячейки ниже, чтобы выполнить их и наблюдать результат. В качестве задачи вам предлагается попробовать этот код с разными данными.

## Цель

В этом уроке мы обсуждали различные концепции, связанные с Data Science. Давайте попробуем обнаружить больше связанных концепций, сделав некоторое **текстовое извлечение**. Мы начнем с текста о Data Science, извлечем из него ключевые слова, а затем попробуем визуализировать результат.

В качестве текста я возьму страницу о Data Science из Википедии:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Шаг 1: Получение данных

Первый шаг в любом процессе работы с данными — получение данных. Мы будем использовать библиотеку `requests` для этого:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Шаг 2: Преобразование данных

Следующий шаг — преобразовать данные в форму, пригодную для обработки. В нашем случае мы скачали исходный код HTML со страницы, и нам нужно преобразовать его в простой текст.

Существует много способов сделать это. Мы будем использовать [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), популярную библиотеку Python для разбора HTML. BeautifulSoup позволяет нам нацеливаться на конкретные HTML-элементы, поэтому мы можем сосредоточиться на основном содержимом статьи с Википедии и убрать некоторые навигационные меню, боковые панели, нижние колонтитулы и другой нерелевантный контент (хотя часть стандартного текста все же может остаться).


Сначала нам нужно установить библиотеку BeautifulSoup для парсинга HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Шаг 3: Получение инсайтов

Самый важный шаг — превратить наши данные в такую форму, из которой мы можем извлекать инсайты. В нашем случае мы хотим извлечь ключевые слова из текста и увидеть, какие ключевые слова более значимы.

Мы будем использовать библиотеку Python под названием [RAKE](https://github.com/aneesha/RAKE) для извлечения ключевых слов. Сначала установим эту библиотеку, если она не установлена: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Основной функционал доступен через объект `Rake`, который мы можем настроить с помощью некоторых параметров. В нашем случае мы установим минимальную длину ключевого слова в 5 символов, минимальную частоту ключевого слова в документе равной 3, и максимальное количество слов в ключевом слове — 2. Не стесняйтесь экспериментировать с другими значениями и наблюдать результат.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Мы получили список терминов вместе с соответствующей степенью важности. Как вы можете видеть, наиболее актуальные дисциплины, такие как машинное обучение и большие данные, присутствуют в списке на верхних позициях.

## Шаг 4: Визуализация результата

Людям проще всего интерпретировать данные в визуальной форме. Поэтому часто имеет смысл визуализировать данные, чтобы получить какие-то выводы. Мы можем использовать библиотеку `matplotlib` в Python для построения простого распределения ключевых слов с их релевантностью:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Однако есть еще более удобный способ визуализации частоты слов - с помощью **Облака слов**. Нам потребуется установить другую библиотеку для построения облака слов из нашего списка ключевых слов.


In [ ]:
!{sys.executable} -m pip install wordcloud

Объект `WordCloud` отвечает за приём исходного текста или заранее вычисленного списка слов с их частотами и возвращает изображение, которое затем можно отобразить с помощью `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Мы также можем передать исходный текст в `WordCloud` — давайте посмотрим, сможем ли получить похожий результат:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Вы можете видеть, что облако слов теперь выглядит более впечатляюще, но оно также содержит много шума (например, нерелевантные слова, такие как `Retrieved on`). Кроме того, мы получаем меньше ключевых слов, состоящих из двух слов, таких как *data scientist* или *computer science*. Это потому, что алгоритм RAKE гораздо лучше справляется с выбором хороших ключевых слов из текста. Этот пример иллюстрирует важность предварительной обработки и очистки данных, так как четкая картина в конце позволит нам принимать более обоснованные решения.

В этом упражнении мы прошли простой процесс извлечения смысла из текста Википедии в виде ключевых слов и облака слов. Этот пример довольно простой, но хорошо демонстрирует все типичные шаги, которые предпринимает специалист по данным при работе с данными, начиная с их получения и заканчивая визуализацией.

В нашем курсе мы подробно обсудим все эти шаги.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
